# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saif-Ullah0/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os, sys, json
import pandas as pd
import numpy as np

# Clone repository if not already present
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/Saif-Ullah0/flyrank-ml-internship.git

# Move into repository directory if sitting outside it
if os.path.exists("flyrank-ml-internship"):
    os.chdir("flyrank-ml-internship")

# Verify current working directory and load data
data_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Directory check failed. Current CWD: {os.getcwd()}")

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
print(df.columns.tolist())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 140 (delta 50), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.86 MiB | 7.89 MiB/s, done.
Resolving deltas: 100% (50/50), done.
Loaded 30000 rows, 44 columns
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_c

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Logic: Combine normalized staleness (days_since_last_update) with log-scaled traffic volume (impressions_90d) to identify high-traffic pages that have not been updated in a long time.

Reason Codes:

STALE_HIGH_TRAFFIC: Score > 0.5 (Needs immediate content refresh)

MODERATE_DECAY_RISK: Score between 0.25 and 0.5 (Monitor regularly)

LOW_PRIORITY: Score < 0.25 (No immediate action)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os, json
import pandas as pd
import numpy as np

# 1. Define binary target if not already created
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "declining").astype(int)

# 2. Feature Normalization (Log scale on impressions prevents outlier skew)
df["staleness_score"] = df["days_since_last_update"] / df["days_since_last_update"].max()
df["visibility_score"] = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())

# 3. Composite Baseline Action Score (60% Staleness, 40% Visibility)
df["baseline_score"] = (0.6 * df["staleness_score"] + 0.4 * df["visibility_score"]).round(4)

# 4. Reason Code & Action Label Assignment
def assign_reason(score):
    if score > 0.5:
        return "STALE_HIGH_TRAFFIC"
    elif score > 0.25:
        return "MODERATE_DECAY_RISK"
    return "LOW_PRIORITY"

df["reason_code"] = df["baseline_score"].apply(assign_reason)
df["action_label"] = df["baseline_score"].apply(
    lambda x: "REFRESH_NOW" if x > 0.5 else ("MONITOR" if x > 0.25 else "LOW_PRIORITY")
)

# 5. Build and Sort Ranked Queue
queue = df[[
    "baseline_score", "reason_code", "action_label",
    "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "is_declining_label"
]].sort_values("baseline_score", ascending=False).reset_index(drop=True)

# 6. Save CSV Output
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

# 7. Evaluate Baseline Precision@50 & Export Metrics
top50 = queue.head(50)
precision_50 = float(top50["is_declining_label"].mean())

metrics = {
    "baseline_precision_at_50": round(precision_50, 3),
    "rule": "0.6 * norm(days_since_last_update) + 0.4 * log_norm(impressions_90d)",
    "total_rows_scored": len(queue)
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Queue written to {output_path} ({len(queue)} rows)")
print(f"Baseline Precision@50: {precision_50:.3f}")
print("First 3 rows of queue:")
print(queue.head(3))

Queue written to work/outputs/baseline_action_score.csv (30000 rows)
Baseline Precision@50: 0.000
First 3 rows of queue:
   baseline_score         reason_code action_label  days_since_last_update  \
0          0.7089  STALE_HIGH_TRAFFIC  REFRESH_NOW                     373   
1          0.6928  STALE_HIGH_TRAFFIC  REFRESH_NOW                     301   
2          0.6882  STALE_HIGH_TRAFFIC  REFRESH_NOW                     301   

   impressions_90d  avg_position   ctr  is_declining_label  
0               35           7.5  0.00                   0  
1              954           9.0  0.42                   0  
2              821           5.8  0.24                   0  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# Select top 20 rows from the ranked queue
top20 = queue.head(20).reset_index(drop=True)

print("TOP-20 REVIEW\n")
print(f"{'#':<4} {'Score':<8} {'Action':<15} {'Reason Code':<20} {'Stale(d)':<10} {'Impr':<10} {'Decl':<6}")
print("-" * 75)

for i, row in top20.iterrows():
    print(f"{i+1:<4} {row['baseline_score']:<8.4f} "
          f"{row['action_label']:<15} "
          f"{row['reason_code']:<20} "
          f"{int(row['days_since_last_update']):<10} "
          f"{int(row['impressions_90d']):<10} "
          f"{int(row['is_declining_label'])}")

TOP-20 REVIEW

#    Score    Action          Reason Code          Stale(d)   Impr       Decl  
---------------------------------------------------------------------------
1    0.7089   REFRESH_NOW     STALE_HIGH_TRAFFIC   373        35         0
2    0.6928   REFRESH_NOW     STALE_HIGH_TRAFFIC   301        954        0
3    0.6882   REFRESH_NOW     STALE_HIGH_TRAFFIC   301        821        0
4    0.6774   REFRESH_NOW     STALE_HIGH_TRAFFIC   313        304        0
5    0.6610   REFRESH_NOW     STALE_HIGH_TRAFFIC   301        335        0
6    0.6608   REFRESH_NOW     STALE_HIGH_TRAFFIC   313        176        0
7    0.6596   REFRESH_NOW     STALE_HIGH_TRAFFIC   335        52         0
8    0.6521   REFRESH_NOW     STALE_HIGH_TRAFFIC   305        202        0
9    0.6511   REFRESH_NOW     STALE_HIGH_TRAFFIC   304        206        0
10   0.6474   REFRESH_NOW     STALE_HIGH_TRAFFIC   194        61678      0
11   0.6463   REFRESH_NOW     STALE_HIGH_TRAFFIC   194        59472      0
12  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# 1. Quantify Weak Picks / False Positives in Top 50
top50 = queue.head(50)
false_positives = top50[top50["is_declining_label"] == 0]
fp_count = len(false_positives)
fp_rate = (fp_count / 50) * 100

print(f"WEAK PICKS ANALYSIS (Top 50)")
print(f"False Positives in Top 50: {fp_count} / 50 ({fp_rate:.1f}%)")
print(f"True Declines (Precision@50): {precision_50:.3f}")
print("-" * 50)

print("\nExamples of Weak Picks (High Score, Zero Decline):")
print(false_positives[["baseline_score", "reason_code", "days_since_last_update", "impressions_90d", "is_declining_label"]].head(5))

# 2. Data Leakage Self-Check
potential_leakage_cols = [
    col for col in df.columns
    if any(k in col for k in ["future", "trend", "declining", "label", "next_"])
]

print("\nDATA LEAKAGE CHECK:")
print(f"Excluded columns with future/label signals: {potential_leakage_cols}")
print("Features used in baseline rule: ['days_since_last_update', 'impressions_90d']")
print("Leakage Status: CONFIRMED CLEAN (No target or future window features used in scoring)")

WEAK PICKS ANALYSIS (Top 50)
False Positives in Top 50: 50 / 50 (100.0%)
True Declines (Precision@50): 0.000
--------------------------------------------------

Examples of Weak Picks (High Score, Zero Decline):
   baseline_score         reason_code  days_since_last_update  \
0          0.7089  STALE_HIGH_TRAFFIC                     373   
1          0.6928  STALE_HIGH_TRAFFIC                     301   
2          0.6882  STALE_HIGH_TRAFFIC                     301   
3          0.6774  STALE_HIGH_TRAFFIC                     313   
4          0.6610  STALE_HIGH_TRAFFIC                     301   

   impressions_90d  is_declining_label  
0               35                   0  
1              954                   0  
2              821                   0  
3              304                   0  
4              335                   0  

DATA LEAKAGE CHECK:
Excluded columns with future/label signals: ['trend_direction', 'trend_pct', 'is_declining_label', 'action_label']
Features used i

Weak Picks & Leakage Verification
Why these picks are weak:
Staleness $\neq$ Decay: A page updated >300 days ago with high visibility might be a stable evergreen piece that continues to rank without updates. Heuristic scoring flags it purely on age, creating high false-positive noise.

Impression Skew: Low-traffic stale pages (e.g., 1–35 impressions) can score high due to raw age weights, wasting operational effort on non-impactful pages.

Lack of Trend Context: The rule evaluates point-in-time state (days_since_last_update and impressions_90d) rather than directional traffic momentum.

Leakage Check Confirmation:Verified that is_declining_label, trend_direction, and trend_pct were not used in calculating baseline_score.The rule relies strictly on historical, point-in-time metadata available at scoring time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.